In [3]:
import numpy as np
import scipy.stats as stats

f) Сгенерируйте выборку объема n = 100 для некоторого значения параметра $\theta$

In [169]:
n = 100
theta_true = 5
beta = 0.95
np.random.seed(46)

sample = stats.pareto.rvs(b=theta_true-1, size=n)

theta_omp = 1 + n / (np.sum(np.log(sample)))

print(f"Выбранное theta: {theta_true}")
print(f"Выборка: {sample}")
print(f"Оценка метода максимального правдоподобия: {theta_omp:.4f}")

Выбранное theta: 5
Выборка: [1.46656841 1.28640384 1.07422745 1.42587088 1.09843149 1.99790436
 1.01101303 1.15643508 1.83981504 1.16386245 1.19397226 1.02243052
 1.14904811 1.40822046 1.65616483 1.10270582 1.02785538 1.49931528
 1.13378577 1.28025535 1.00920513 1.09322914 1.01228131 1.12343148
 1.07906352 3.20530249 1.13393261 1.09622881 1.06446095 1.18568067
 1.015016   1.08928006 1.06924592 2.28675687 1.27006229 1.6424443
 1.0238818  1.04115693 1.32197131 2.62665742 1.00589164 1.81433732
 1.01506345 1.42234114 1.01683619 1.07663346 1.04756066 1.05576728
 1.02856862 1.20190029 1.00156769 1.47828603 1.03803684 1.02873802
 1.15068206 1.06373505 1.57105331 1.0014741  1.04095333 1.34934173
 1.04073558 1.66790526 1.19767188 1.14827347 1.00479913 1.18585702
 2.08048101 1.11191844 1.13971578 1.55490647 1.36745907 1.18135402
 1.06847652 1.32138806 1.24119626 1.16035565 1.23962398 1.09764149
 1.30204938 1.02412538 1.1212236  1.23646689 1.51607262 1.17012798
 1.2772834  1.22620425 1.10623507 1

Вычислите указанные выше доверительные интервалы для доверительной вероятности 0.95

In [170]:
# b) Ассимптотический доверительный интервал для медианы
t1 = stats.norm.ppf((1 - beta) / 2)
t2 = stats.norm.ppf((1 + beta) / 2)

start = 2 ** (1 / (theta_omp - 1)) - (2 ** (1 / (theta_omp - 1)) * np.log(2)) / (n ** 0.5) / (theta_omp - 1) * t2
end = 2 ** (1 / (theta_omp - 1)) - (2 ** (1 / (theta_omp - 1)) * np.log(2)) / (n ** 0.5) / (theta_omp - 1) * t1
l =  end - start
print(f"Медиана: {2 ** (1 / (theta_true - 1)):.4f}")
print(f"Оценка медианы: {2 ** (1 / (theta_omp - 1)):.4f}")
print(f"Ассимптотический доверительный интервал для медианы: ({start:.4f}, {end:.4f}), l = {l:.4f}")

# c) Асимптотический доверительный интервал (на основе ОМП)
start_omp = theta_omp - n ** 0.5 / np.sum(np.log(sample)) * t2
end_omp = theta_omp - n ** 0.5 / np.sum(np.log(sample)) * t1
l_omp = end_omp - start_omp
print(f"Асимптотический доверительный интервал (на основе ОМП): ({start_omp:.4f}, {end_omp:.4f}), l = {l_omp:.4f}")

Медиана: 1.1892
Оценка медианы: 1.1534
Ассимптотический доверительный интервал для медианы: (1.1212, 1.1857), l = 0.0645
Асимптотический доверительный интервал (на основе ОМП): (4.9040, 6.8074), l = 1.9034


t) Численно постройте бутстраповский доверительный интервал двумя способами, используя параметрический и непараметрический бутстрапы

In [171]:
# непараметрический бутстрап
k = 1000
bootstrap_estimates = []
for _ in range(k):
    boot_sample = np.random.choice(sample, size=n, replace=True)
    boot_theta = 1 + n / np.sum(np.log(boot_sample))
    bootstrap_estimates.append(boot_theta - theta_omp)

bootstrap_estimates = np.array(sorted(bootstrap_estimates))
k1 = int((1 - beta) / 2 * k - 1)
k2 = int((1 + beta) / 2 * k - 1)

start_boot_not_param = theta_omp - bootstrap_estimates[k2]
end_boot_not_param = theta_omp - bootstrap_estimates[k1]
l_boot_not_param = end_boot_not_param - start_boot_not_param
print(f"Непараметрический бутстраповский доверительный интервал: ({start_boot_not_param:.4f}, {end_boot_not_param:.4f}), l = {l_boot_not_param:.4f}")


Непараметрический бутстраповский доверительный интервал: (4.6715, 6.7025), l = 2.0310


In [172]:
# параметрический бутстрап
k = 50000
bootstrap_estimates = []
for _ in range(k):
    boot_sample = stats.pareto.rvs(b=theta_omp - 1, size=n)
    boot_theta = 1 + n / np.sum(np.log(boot_sample))
    bootstrap_estimates.append(boot_theta - theta_omp)

bootstrap_estimates = np.array(sorted(bootstrap_estimates))
k1 = int((1 - beta) / 2 * k - 1)
k2 = int((1 + beta) / 2 * k - 1)

start_boot_param = theta_omp - bootstrap_estimates[k2]
end_boot_param = theta_omp - bootstrap_estimates[k1]
l_boot_param = end_boot_param - start_boot_param
print(f"Параметрический бутстраповский доверительный интервал: ({start_boot_param:.4f}, {end_boot_param:.4f}), l = {l_boot_param:.4f}")

Параметрический бутстраповский доверительный интервал: (4.7507, 6.6897), l = 1.9390


h) Сравнить все интервалы

In [173]:
intervals = [("Ассимптотический (ОМП)", l_omp),
             ("Непараметрический бутстрап", l_boot_not_param),
             ("Параметрический бутстрап", l_boot_param)]
intervals = sorted(intervals, key=lambda x: x[1])

for name, length in intervals:
    print(f"{name}: l = {length:.4f}")

Ассимптотический (ОМП): l = 1.9034
Параметрический бутстрап: l = 1.9390
Непараметрический бутстрап: l = 2.0310
